# Self-Refine — GSM-8k Math Reasoning (Novelty Edition)

## Architecture Overview
| Stage | Model | Role |
|---|---|---|
| **Model 1** | `llama-3.1-8b-instant` | Initial solution generation |
| **Model 2** | `gemma2-9b-it` | Dual-feedback generation + ranking |
| **Model 3** | `mixtral-8x7b-32768` | Solution refinement |

### Novelty Contributions
1. **Specialist models** — each stage uses the model best suited to its task  
2. **Dual-feedback ranking** — two independent feedbacks generated; Model 2 picks the higher-quality one before refinement  
3. **Cross-model quota** — by spreading calls across three free-tier models, effective per-model rate limits triple  
4. **3-curve comparison** — Paper (GPT-3.5) · Groq Baseline (single model) · Novelty (3-model)


In [ ]:
# ── Step 1: Clone Self-Refine repo and install dependencies ───────────────────
!git clone https://github.com/madaan/self-refine 2>/dev/null || echo "Repo already cloned"
!pip install -q groq transformers
!pip show groq | grep -E "Name|Version"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.4 MB/s eta 0:00:00
Name: groq
Version: 1.2.0


In [ ]:
import sys, os, json, time, re, pickle
from collections import defaultdict
sys.path.append("/content/self-refine")
from groq import Groq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print("✅ All imports successful")


✅ All imports successful


In [ ]:
# ── API Key ────────────────────────────────────────────────────────────────────
GROQ_API_KEY = "#####"   # 🔑 Replace with your Groq API key

# ── Three specialist models (all available on Groq free tier) ──────────────────
MODEL_INIT     = "llama-3.1-8b-instant"   # Model 1 – fast, great for code gen
MODEL_FEEDBACK = "llama-3.3-70b-versatile"           # Model 2 – analytical, good evaluator
MODEL_REFINE   = "meta-llama/llama-4-scout-17b-16e-instruct"     # Model 3 – strong reasoning for fixes

# ── Single model baseline (original approach) ──────────────────────────────────
MODEL_BASELINE = "llama-3.1-8b-instant"

client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client ready")
print(f"   INIT     → {MODEL_INIT}")
print(f"   FEEDBACK → {MODEL_FEEDBACK}")
print(f"   REFINE   → {MODEL_REFINE}")


✅ Groq client ready
   INIT     → llama-3.1-8b-instant
   FEEDBACK → llama-3.3-70b-versatile
   REFINE   → meta-llama/llama-4-scout-17b-16e-instruct


In [ ]:
# ── Quick smoke test for all three models ──────────────────────────────────────
for tag, model in [("INIT", MODEL_INIT), ("FEEDBACK", MODEL_FEEDBACK), ("REFINE", MODEL_REFINE)]:
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Reply with exactly: WORKING"}],
            max_tokens=10
        )
        print(f"✅ {tag:8s} ({model}): {r.choices[0].message.content.strip()}")
    except Exception as e:
        print(f"❌ {tag:8s} ({model}): {e}")
    time.sleep(2)


✅ INIT     (llama-3.1-8b-instant): WORKING
✅ FEEDBACK (llama-3.3-70b-versatile): WORKING
✅ REFINE   (meta-llama/llama-4-scout-17b-16e-instruct): WORKING


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PROMPT TEMPLATES  (identical to original Self-Refine paper prompts)
# ══════════════════════════════════════════════════════════════════════════════

TASK_INIT_PROMPT = """# Q: There were nine computers in the server room. Five more
# computers were installed each day, from monday to thursday.
# How many computers are now in the server room?

# solution using Python:
def solution():
    \"\"\"...\"\"\"
    computers_initial = 9
    computers_per_day = 5
    num_days = 4  # 4 days between monday and thursday
    computers_added = computers_per_day * num_days
    computers_total = computers_initial + computers_added
    return computers_total

###

# Q: Shawn has five toys. For Christmas, he got two toys each from
# his mom and dad. How many toys does he have now?

# solution using Python:
def solution():
    \"\"\"...\"\"\"
    toys_initial = 5
    mom_toys = 2
    dad_toys = 2
    total_toys = toys_initial + mom_toys + dad_toys
    return total_toys

###

# Q: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason
# has 12 lollipops. How many lollipops did Jason give to Denny?

# solution using Python:
def solution():
    \"\"\"...\"\"\"
    jason_initial = 20
    jason_after = 12
    denny_lollipops = jason_initial - jason_after
    return denny_lollipops

###

# Q: {question}

# solution using Python:
"""

FEEDBACK_PROMPT = """def solution():
    \"\"\"Twenty dozen cups cost $1200 less than the total cost of half
    a dozen plates sold at $6000 each. Total cost per cup?\"\"\"
    plates = 6
    plate_cost = 6000
    cups = 12 * 20
    cup_cost = plate_cost   # BUG: cup cost != plate cost
    return cup_cost

# Find the error by checking each block step-by-step:
plates = 6; plate_cost = 6000  # looks good
cups = 12 * 20                 # looks good
cup_cost = plate_cost
# WRONG! cup_cost is $1200 less than total plate cost, not equal to plate_cost

###

def solution():
    \"{question}\"
{body}

# Find the error by checking each block step-by-step:
"""

REFINE_PROMPT = """def solution():
    \"\"\"Twenty dozen cups cost $1200 less than the total cost of half
    a dozen plates sold at $6000 each. Total cost per cup?\"\"\"
    plates = 6
    plate_cost = 6000
    cups = 12 * 20
    cup_cost = plate_cost   # WRONG
    return cup_cost

# Error: cup_cost should be (plate_cost * plates - 1200) / cups

Okay! Here is the rewrite:
def solution():
    \"\"\"...\"\"\"
    plates = 6
    plate_cost = 6000
    cups = 12 * 20
    total_cup_cost = (plate_cost * plates) - 1200
    return total_cup_cost / cups

###

def solution():
    \"{question}\"
{body}

# {feedback}

Okay! Here is the rewrite:
"""

# ── NEW: Feedback ranking prompt (used by Model 2) ─────────────────────────────
FEEDBACK_RANK_PROMPT = """You are an expert code reviewer comparing two pieces of feedback for a Python math solution.

Math question: {question}

Current Python solution:
{code}

=== Feedback A ===
{feedback_a}

=== Feedback B ===
{feedback_b}

Task: Which feedback will be MORE USEFUL for correcting the Python solution?
Evaluate based on: accuracy of error identification, specificity, and actionability.

Reply with ONLY the letter A or B (no explanation, no punctuation — just one letter)."""

print("✅ All prompt templates loaded (TASK_INIT, FEEDBACK, REFINE, FEEDBACK_RANK)")


✅ All prompt templates loaded (TASK_INIT, FEEDBACK, REFINE, FEEDBACK_RANK)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CORE HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def call_groq_model(prompt, model, system=None, max_tokens=600,
                    temperature=0.0, retries=4):
    """Universal Groq caller — supports any model, configurable temperature."""
    if system is None:
        system = ("You are an expert Python programmer solving math word problems. "
                  "ALWAYS respond with raw Python only — no markdown, no ``` fences, no prose. "
                  "Output exactly one function named solution() and nothing else.")
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system},
                          {"role": "user",   "content": prompt}],
                max_tokens=max_tokens,
                temperature=temperature
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if "429" in err and attempt < retries - 1:
                wait = 35 * (attempt + 1)
                print(f"  ⏳ Rate limit [{model}] — waiting {wait}s (attempt {attempt+1})…")
                time.sleep(wait)
            else:
                print(f"  ❌ Error [{model}]: {e}")
                return ""
    return ""


def extract_solution(text):
    """Pull out def solution(): block; strips markdown fences first."""
    text = re.sub(r'```(?:python)?\n?', '', text)
    text = text.replace('```', '')
    if 'def solution():' in text:
        idx = text.index('def solution():')
        code = text[idx:].split('\n###')[0].strip()
        return code
    return text.strip()


def get_body(code):
    """Return indented body lines of solution() (skip the def line)."""
    lines = extract_solution(code).split('\n')
    body, in_fn = [], False
    for line in lines:
        if line.strip().startswith('def solution'): in_fn = True; continue
        if in_fn: body.append(line)
    return '\n'.join(body) or '    pass'


def execute(code):
    """Run solution() and return float result, or None on error."""
    try:
        code = extract_solution(code)
        ns = {}; exec(code, ns)
        if 'solution' in ns:
            r = ns['solution']()
            return float(r) if r is not None else None
    except: pass
    return None


def parse_gt(s):
    """Parse ground-truth answer (handles int, float, '####' strings, commas)."""
    try:
        if isinstance(s, (int, float)): return float(s)
        if '####' in s: s = s.split('####')[-1]
        return float(s.replace(',', '').strip())
    except: return None


def is_correct(pred, gt, tol=0.01):
    if pred is None or gt is None: return False
    if gt == 0: return abs(pred) < tol
    return abs(pred - gt) / abs(gt) < tol


def build_accuracy(all_res, max_iter):
    """Return (iterations_list, accuracy_list) from experiment results."""
    correct_cnt = defaultdict(int)
    total_cnt   = defaultdict(int)
    for ex in all_res:
        for d in ex['iters']:
            it = d['iter']
            total_cnt[it]  += 1
            correct_cnt[it] += int(d['correct'])
    iters = sorted(total_cnt.keys())
    accs  = [correct_cnt[it] / total_cnt[it] * 100 for it in iters]
    return iters, accs


print("✅ Helper functions ready")


✅ Helper functions ready


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# NOVELTY  (3-model pipeline + dual-feedback ranking)
# ══════════════════════════════════════════════════════════════════════════════

# ── Model 1: Initial Generation ───────────────────────────────────────────────
def init_solution_novel(question):
    """
    MODEL_INIT generates the first draft Python solution.
    llama-3.1-8b-instant is fast and reliable for structured code generation.
    """
    prompt = TASK_INIT_PROMPT.format(question=question)
    return extract_solution(
        call_groq_model(prompt, MODEL_INIT, max_tokens=500)
    )


# ── Model 2 helper: single feedback with configurable temperature ─────────────
def _get_one_feedback(question, code, temperature=0.0):
    """Internal helper — generate one feedback from MODEL_FEEDBACK."""
    body   = get_body(code)
    prompt = FEEDBACK_PROMPT.format(question=question, body=body)
    for attempt in range(4):
        try:
            resp = client.chat.completions.create(
                model=MODEL_FEEDBACK,
                messages=[
                    {"role": "system", "content": (
                        "You are a math teacher. Check Python solutions line-by-line. "
                        "If correct, say \'The output is good.\' "
                        "Otherwise identify the exact bug. No markdown, plain text only.")},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=400,
                temperature=temperature
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                wait = 35 * (attempt + 1)
                print(f"    ⏳ Rate limit [FEEDBACK model] — waiting {wait}s…")
                time.sleep(wait)
            else:
                print(f"    ❌ Feedback error: {e}")
                return ""
    return ""


# ── Model 2: Dual Feedback + Ranking ──────────────────────────────────────────
def get_dual_feedback_ranked(question, code):
    """
    NOVELTY CORE:
      1. Generate Feedback A  (temperature=0.0 — deterministic, analytical)
      2. Generate Feedback B  (temperature=0.4 — slightly creative)
      3. Ask MODEL_FEEDBACK to rank A vs B and return the winner
    Returns: (best_feedback_text, winner_label)
    """
    print("    📝 [Model 2] Generating Feedback A (temp=0.0) …")
    fb_a = _get_one_feedback(question, code, temperature=0.0)
    time.sleep(2)   # inter-call buffer to respect rate limits

    print("    📝 [Model 2] Generating Feedback B (temp=0.4) …")
    fb_b = _get_one_feedback(question, code, temperature=0.4)
    time.sleep(2)

    # If one feedback is empty or identical, skip ranking
    if not fb_b or fb_a.strip() == fb_b.strip():
        print("    🏆 Skipped ranking — using Feedback A")
        return fb_a, "A"

    # Ranking call
    print("    ⚖️  [Model 2] Ranking feedbacks …")
    rank_prompt = FEEDBACK_RANK_PROMPT.format(
        question=question,
        code=extract_solution(code),
        feedback_a=fb_a,
        feedback_b=fb_b
    )
    raw = call_groq_model(
        rank_prompt, MODEL_FEEDBACK,
        system="You are an expert code reviewer. Reply with ONLY the letter A or B.",
        max_tokens=5,
        temperature=0.0
    )
    time.sleep(1)

    winner = "A" if "A" in raw.upper() else "B"
    best_fb = fb_a if winner == "A" else fb_b
    print(f"    🏆 Winner: Feedback {winner}")
    return best_fb, winner


# ── Model 3: Refinement ────────────────────────────────────────────────────────
def refine_solution_novel(question, code, feedback):
    """
    MODEL_REFINE (mixtral-8x7b) uses Mixtral's strong reasoning to produce
    a corrected solution guided by the ranked feedback.
    """
    body   = get_body(code)
    prompt = REFINE_PROMPT.format(question=question, body=body, feedback=feedback)
    return extract_solution(
        call_groq_model(
            prompt, MODEL_REFINE, max_tokens=500,
            system=("You are an expert Python programmer. "
                    "Rewrite the solution to fix the identified error. "
                    "Output raw Python only — one function named solution(), no markdown.")
        )
    )


print("✅ Novelty functions ready")
print(f"   Model 1 (INIT)     → {MODEL_INIT}")
print(f"   Model 2 (FEEDBACK) → {MODEL_FEEDBACK}  [dual + ranking]")
print(f"   Model 3 (REFINE)   → {MODEL_REFINE}")


✅ Novelty functions ready
   Model 1 (INIT)     → llama-3.1-8b-instant
   Model 2 (FEEDBACK) → llama-3.3-70b-versatile  [dual + ranking]
   Model 3 (REFINE)   → meta-llama/llama-4-scout-17b-16e-instruct


In [ ]:
# ── Smoke test: single question through the full novelty pipeline ──────────────
Q  = ("There were nine computers in the server room. "
      "Five more computers were installed each day, from monday to thursday. "
      "How many computers are now in the server room?")
GT = 29.0

print("=" * 65)
print("🧪  NOVELTY SMOKE TEST")
print("=" * 65)
print(f"Q : {Q}")
print(f"GT: {GT}\n")

# Model 1 — initial
code0 = init_solution_novel(Q)
print("── [Model 1] Initial solution ──")
print(code0)
print(f"→ result: {execute(code0)}  correct: {is_correct(execute(code0), GT)}\n")
time.sleep(1)

# Model 2 — dual feedback + ranking
best_fb, winner = get_dual_feedback_ranked(Q, code0)
print(f"\n── [Model 2] Best feedback (winner={winner}) ──")
print(best_fb[:300])
time.sleep(1)

# Model 3 — refinement
code1 = refine_solution_novel(Q, code0, best_fb)
print("\n── [Model 3] Refined solution ──")
print(code1)
print(f"→ result: {execute(code1)}  correct: {is_correct(execute(code1), GT)}")


🧪  NOVELTY SMOKE TEST
Q : There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?
GT: 29.0

── [Model 1] Initial solution ──
def solution():
    computers_initial = 9
    computers_per_day = 5
    num_days = 4  # 4 days between monday and thursday
    computers_added = computers_per_day * num_days
    computers_total = computers_initial + computers_added
    return computers_total
→ result: 29.0  correct: True

    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B

── [Model 2] Best feedback (winner=B) ──
The given code for the second problem looks good, let's check it step-by-step: 
computers_initial = 9  # looks good
computers_per_day = 5  # looks good
num_days = 4  # looks good, there are indeed 4 days from monday to thursday
computers_added = computers_per_day * num_d

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VARIABLES
# ══════════════════════════════════════════════════════════════════════════════

GSM_FILE         = "/content/self-refine/data/tasks/gsm/gsm.jsonl"
N_EXAMPLES       = 50    # number of GSM-8k problems to evaluate
MAX_ITER         = 4     # max Self-Refine iterations per problem
NOVELTY_PKL      = "/content/novelty_results.pkl"

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT — NOVELTY  (3 specialist models + dual-feedback ranking)
# ══════════════════════════════════════════════════════════════════════════════

def run_novelty(data_file, n, max_iter):
    all_res, examples = [], []
    with open(data_file) as f:
        for i, line in enumerate(f):
            if i >= n: break
            if line.strip(): examples.append(json.loads(line.strip()))

    print(f"📊 NOVELTY | 3-Model + Dual-Feedback | {len(examples)} examples | k={max_iter}")
    print(f"   Model 1 (INIT)     → {MODEL_INIT}")
    print(f"   Model 2 (FEEDBACK) → {MODEL_FEEDBACK}")
    print(f"   Model 3 (REFINE)   → {MODEL_REFINE}")

    for idx, ex in enumerate(examples):
        question = ex.get("input", ex.get("question", ""))
        gt       = parse_gt(ex.get("target", ex.get("answer", "")))
        print(f"\n📌 [{idx+1}/{len(examples)}] GT={gt} | {question[:65]}…")

        rec = {"question": question, "gt": gt, "iters": [], "fb_winners": []}

        # ── Iter 0: Model 1 ───────────────────────────────────────────────────
        print(f"  🤖 [Model 1] Generating initial solution…")
        code = init_solution_novel(question)
        res  = execute(code)
        ok   = is_correct(res, gt)
        rec["iters"].append({"iter": 0, "code": code, "result": res, "correct": ok, "fb": None})
        print(f"  {'✅' if ok else '❌'} iter 0  pred={res}")
        time.sleep(1)

        for it in range(1, max_iter + 1):
            if ok:
                # Early stop: propagate correct answer through remaining iters
                for r in range(it, max_iter + 1):
                    rec["iters"].append({"iter": r, "code": code, "result": res,
                                         "correct": ok, "fb": "stopped (correct)"})
                break

            # ── Model 2: dual feedback + ranking ──────────────────────────────
            print(f"  🤖 [Model 2] Iter {it} — dual feedback…")
            best_fb, winner = get_dual_feedback_ranked(question, code)
            rec["fb_winners"].append(winner)

            # ── Model 3: refinement ───────────────────────────────────────────
            print(f"  🤖 [Model 3] Iter {it} — refining with feedback {winner}…")
            code = refine_solution_novel(question, code, best_fb)
            time.sleep(1)
            res  = execute(code)
            ok   = is_correct(res, gt)
            rec["iters"].append({"iter": it, "code": code, "result": res, "correct": ok, "fb": best_fb})
            print(f"  {'✅' if ok else '❌'} iter {it}  pred={res}")
            time.sleep(1)

        all_res.append(rec)
        with open(NOVELTY_PKL, "wb") as f: pickle.dump(all_res, f)

    print("\n✅ Novelty experiment complete — saved to", NOVELTY_PKL)
    return all_res


novelty_results = run_novelty(GSM_FILE, N_EXAMPLES, MAX_ITER)


📊 NOVELTY | 3-Model + Dual-Feedback | 50 examples | k=4
   Model 1 (INIT)     → llama-3.1-8b-instant
   Model 2 (FEEDBACK) → llama-3.3-70b-versatile
   Model 3 (REFINE)   → meta-llama/llama-4-scout-17b-16e-instruct

📌 [1/50] GT=18.0 | Janet’s ducks lay 16 eggs per day. She eats three for breakfast e…
  🤖 [Model 1] Generating initial solution…
  ✅ iter 0  pred=18.0

📌 [2/50] GT=3.0 | A robe takes 2 bolts of blue fiber and half that much white fiber…
  🤖 [Model 1] Generating initial solution…
  ✅ iter 0  pred=3.0

📌 [3/50] GT=70000.0 | Josh decides to try flipping a house.  He buys a house for $80,00…
  🤖 [Model 1] Generating initial solution…
  ❌ iter 0  pred=195000.0
  🤖 [Model 2] Iter 1 — dual feedback…
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  🤖 [Model 3] Iter 1 — refining with feedback A…
  ❌ iter 1  pred=195000.0
  🤖 [Model 2] Iter 2 — dual feedback…
    📝 [M

In [ ]:
# ── Accuracy report for novelty ───────────────────────────────────────────────
novel_iters, novel_accs = build_accuracy(novelty_results, MAX_ITER)

print("\n" + "=" * 65)
print("📊  NOVELTY ACCURACY  (3-model + dual-feedback ranking)")
print("=" * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}{'Δ vs base'}")
print("-" * 55)

nc = defaultdict(int); nt = defaultdict(int)
for ex in novelty_results:
    for d in ex["iters"]:
        nt[d["iter"]] += 1
        nc[d["iter"]] += int(d["correct"])

novel0 = nc[0] / nt[0] * 100 if nt[0] else 0
for it in novel_iters:
    acc   = nc[it] / nt[it] * 100
    delta = "(base)" if it == 0 else f"+{acc-novel0:.1f}%"
    label = "Base" if it == 0 else f"Iter {it}"
    print(f"{label:<10}{nc[it]:<10}{nt[it]:<10}{acc:.1f}%    {delta}")

print("-" * 55)

# Feedback winner statistics
all_winners = [w for ex in novelty_results for w in ex.get("fb_winners", [])]
if all_winners:
    a_wins = all_winners.count("A")
    b_wins = all_winners.count("B")
    print(f"\n🏆 Feedback ranking stats:")
    print(f"   Feedback A (temp=0.0) won: {a_wins}/{len(all_winners)} ({a_wins/len(all_winners)*100:.0f}%)")
    print(f"   Feedback B (temp=0.4) won: {b_wins}/{len(all_winners)} ({b_wins/len(all_winners)*100:.0f}%)")



📊  NOVELTY ACCURACY  (3-model + dual-feedback ranking)
Iter      Correct   Total     Accuracy    Δ vs base
-------------------------------------------------------
Base      34        50        68.0%    (base)
Iter 1    48        50        96.0%    +28.0%
Iter 2    48        50        96.0%    +28.0%
Iter 3    48        50        96.0%    +28.0%
Iter 4    48        50        96.0%    +28.0%
-------------------------------------------------------

🏆 Feedback ranking stats:
   Feedback A (temp=0.0) won: 15/22 (68%)
   Feedback B (temp=0.4) won: 7/22 (32%)


In [ ]:
def run_novelty_slice(data_file, start_idx, end_idx, max_iter):
    all_res, examples = [], []
    with open(data_file) as f:
        for i, line in enumerate(f):
            if i < start_idx: continue
            if i >= end_idx: break
            if line.strip(): examples.append(json.loads(line.strip()))

    print(f"📊 NOVELTY SLICE | Samples {start_idx}-{end_idx} | k={max_iter}")
    novelty_slice_pkl = f"/content/novelty_results_{start_idx}_{end_idx}.pkl"

    for idx, ex in enumerate(examples):
        abs_idx = start_idx + idx
        question = ex.get("input", ex.get("question", ""))
        gt       = parse_gt(ex.get("target", ex.get("answer", "")))
        print(f"\n📌 [{abs_idx+1}/{end_idx}] GT={gt} | {question[:60]}...")

        rec = {"question": question, "gt": gt, "iters": [], "fb_winners": []}
        code = init_solution_novel(question)
        res  = execute(code)
        ok   = is_correct(res, gt)
        rec["iters"].append({"iter": 0, "code": code, "result": res, "correct": ok, "fb": None})
        print(f"  {'✅' if ok else '❌'} iter 0  pred={res}")
        time.sleep(1)

        for it in range(1, max_iter + 1):
            if ok:
                for r in range(it, max_iter + 1):
                    rec["iters"].append({"iter": r, "code": code, "result": res, "correct": ok, "fb": "stopped (correct)"})
                break

            best_fb, winner = get_dual_feedback_ranked(question, code)
            rec["fb_winners"].append(winner)
            code = refine_solution_novel(question, code, best_fb)
            time.sleep(1)
            res  = execute(code)
            ok   = is_correct(res, gt)
            rec["iters"].append({"iter": it, "code": code, "result": res, "correct": ok, "fb": best_fb})
            print(f"  {'✅' if ok else '❌'} iter {it}  pred={res}")
            time.sleep(1)

        all_res.append(rec)
        with open(novelty_slice_pkl, "wb") as f: pickle.dump(all_res, f)

    return all_res

# Run for samples 51 to 100
novelty_results_51_100 = run_novelty_slice(GSM_FILE, 50, 100, MAX_ITER)

📊 NOVELTY SLICE | Samples 50-100 | k=4

📌 [51/100] GT=294.0 | Lloyd has an egg farm. His chickens produce 252 eggs per day...
  ✅ iter 0  pred=294.0

📌 [52/100] GT=5.0 | Tom's ship can travel at 10 miles per hour.  He is sailing f...
  ❌ iter 0  pred=1.6666666666666667
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ❌ iter 1  pred=24.499999999999996
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 2  pred=5.0

📌 [53/100] GT=15.0 | Uriah's book bag is getting too heavy for him. He needs to r...
  ✅ iter 0  pred=15.0

📌 [54/100] GT=40.0 | A mechanic charges different rates to repair the tires of tr...
  ✅ iter 0  pred=40.0

📌 [55/100] GT=40.0 | The Doubtfire sisters are driving home with 7 kittens adopte...
  ✅ iter 0  pred=40.0

📌 [56/100] GT=14.0

In [ ]:
# ── Accuracy report for novelty (Samples 51-100) ───────────────
slice_iters, slice_accs = build_accuracy(novelty_results_51_100, MAX_ITER)

print("\n" + "=" * 65)
print("┃ NOVELTY SLICE ACCURACY (Samples 51-100)")
print("=" * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print("-" * 45)

for i, it in enumerate(slice_iters):
    acc = slice_accs[i]
    label = "Base" if it == 0 else f"Iter {it}"
    print(f"{label:<10}{int(acc*len(novelty_results_51_100)/100):<10}{len(novelty_results_51_100):<10}{acc:.1f}%")

print("-" * 45)


┃ NOVELTY SLICE ACCURACY (Samples 51-100)
Iter      Correct   Total     Accuracy    
---------------------------------------------
Base      42        50        84.0%
Iter 1    46        50        92.0%
Iter 2    49        50        98.0%
Iter 3    49        50        98.0%
Iter 4    49        50        98.0%
---------------------------------------------


In [ ]:
combined_results = novelty_results + novelty_results_51_100
combined_iters, combined_accs = build_accuracy(combined_results, MAX_ITER)

print("=" * 50)
print(f"   ACCURACY TABLE: NOVELTY PIPELINE (N={len(combined_results)})")
print("=" * 50)
print(f"{'Iteration':<15} | {'Correct':<10} | {'Accuracy':<10}")
print("-" * 50)

for i, it in enumerate(combined_iters):
    label = "Base (Iter 0)" if it == 0 else f"Iteration {it}"
    correct = int((combined_accs[i] / 100) * len(combined_results))
    print(f"{label:<15} | {correct:<10} | {combined_accs[i]:.1f}%")

print("=" * 50)
print(f"Total Gain: {combined_accs[-1] - combined_accs[0]:+.1f}%")

   ACCURACY TABLE: NOVELTY PIPELINE (N=100)
Iteration       | Correct    | Accuracy  
--------------------------------------------------
Base (Iter 0)   | 76         | 76.0%
Iteration 1     | 94         | 94.0%
Iteration 2     | 97         | 97.0%
Iteration 3     | 97         | 97.0%
Iteration 4     | 97         | 97.0%
Total Gain: +21.0%


In [ ]:
# Run for samples 101 to 150
novelty_results_101_150 = run_novelty_slice(GSM_FILE, 100, 150, MAX_ITER)

📊 NOVELTY SLICE | Samples 100-150 | k=4

📌 [101/150] GT=175.0 | Jerome had 4 friends who came to visit him on a certain day....
  ✅ iter 0  pred=175.0

📌 [102/150] GT=6.0 | Solo has to read 4 pages from his Science textbook, 20 pages...
  ✅ iter 0  pred=6.0

📌 [103/150] GT=26.0 | John likes to have a glass of water with breakfast, lunch an...
  ❌ iter 0  pred=20.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=26.0

📌 [104/150] GT=140.0 | A fog bank rolls in from the ocean to cover a city. It takes...
  ❌ iter 0  pred=14.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=140.0

📌 [105/150] GT=500.0 | Poppy is solving a 1000-piece jigsaw puzzle. She places a qu...
  ✅ iter 0  pred=500.0

📌 [106/150] GT=20.0 | Cody eats thre

In [ ]:
# ── Accuracy report for novelty (Samples 101-150) ───────────────
slice_150_iters, slice_150_accs = build_accuracy(novelty_results_101_150, MAX_ITER)

print("\n" + "=" * 65)
print("┃ NOVELTY SLICE ACCURACY (Samples 101-150)")
print("=" * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print("-" * 45)

for i, it in enumerate(slice_150_iters):
    acc = slice_150_accs[i]
    label = "Base" if it == 0 else f"Iter {it}"
    print(f"{label:<10}{int(acc*len(novelty_results_101_150)/100):<10}{len(novelty_results_101_150):<10}{acc:.1f}%")

print("-" * 45)


┃ NOVELTY SLICE ACCURACY (Samples 101-150)
Iter      Correct   Total     Accuracy    
---------------------------------------------
Base      40        50        80.0%
Iter 1    49        50        98.0%
Iter 2    49        50        98.0%
Iter 3    50        50        100.0%
Iter 4    50        50        100.0%
---------------------------------------------


In [ ]:
final_combined_results = novelty_results + novelty_results_51_100 + novelty_results_101_150
final_iters, final_accs = build_accuracy(final_combined_results, MAX_ITER)

print("=" * 55)
print(f"   CONSOLIDATED ACCURACY TABLE (N={len(final_combined_results)})")
print("=" * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print("-" * 55)

for i, it in enumerate(final_iters):
    label = "Base (Iter 0)" if it == 0 else f"Iteration {it}"
    correct_count = int(round((final_accs[i] / 100) * len(final_combined_results)))
    print(f"{label:<18} | {correct_count:<10} | {final_accs[i]:.1f}%")

print("=" * 55)
print(f"Overall Improvement: {final_accs[-1] - final_accs[0]:+.1f}%")

   CONSOLIDATED ACCURACY TABLE (N=150)
Iteration          | Correct    | Accuracy  
-------------------------------------------------------
Base (Iter 0)      | 116        | 77.3%
Iteration 1        | 143        | 95.3%
Iteration 2        | 146        | 97.3%
Iteration 3        | 147        | 98.0%
Iteration 4        | 147        | 98.0%
Overall Improvement: +20.7%


In [ ]:
import pickle
import os

# Resume from index 162 (Sample 163) to 200
novelty_results_163_200 = run_novelty_slice(GSM_FILE, 162, 200, MAX_ITER)

# Merge with the successful results from the previous interrupted run (151-162)
# We'll reload the partial results if they were saved, or combine if available in memory
if 'novelty_results_151_200' in globals():
    # filter out any incomplete entries from the interrupted run
    successful_151_162 = [r for r in novelty_results_151_200 if len(r['iters']) > 0]
    novelty_results_151_200_final = successful_151_162 + novelty_results_163_200
else:
    novelty_results_151_200_final = novelty_results_163_200

# Save the complete batch
with open('/content/novelty_results_151_200_complete.pkl', 'wb') as f:
    pickle.dump(novelty_results_151_200_final, f)

print(f'\n✅ Completed batch 151-200. Total processed in this segment: {len(novelty_results_151_200_final)}')

📊 NOVELTY SLICE | Samples 162-200 | k=4

📌 [163/200] GT=92.0 | If a bag of marbles costs $20 and the price increases by 20%...
  ❌ iter 0  pred=532.4666656177045
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ❌ Error [meta-llama/llama-4-scout-17b-16e-instruct]: Error code: 503 - {'error': {'message': 'meta-llama/llama-4-scout-17b-16e-instruct is currently over capacity. Please try again and back off exponentially. Visit https://groqstatus.com to see if there is an active incident.', 'type': 'internal_server_error'}}
  ❌ iter 1  pred=None
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ❌ iter 2  pred=532.4666656177045
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedback

In [ ]:
import json

# Diagnostic: Check how many samples are in each list
len_pre_crash = len(novelty_results_151_200) if 'novelty_results_151_200' in globals() else 0
len_post_crash = len(novelty_results_163_200) if 'novelty_results_163_200' in globals() else 0

print(f"Pre-crash list (151-162 attempt) count: {len_pre_crash}")
print(f"Post-crash list (163-200 attempt) count: {len_post_crash}")

# Find indices present in final_200_results between 150 and 200
found_questions = [r['question'] for r in final_200_results]

missing_indices = []
with open(GSM_FILE) as f:
    for i, line in enumerate(f):
        if 150 <= i < 200:
            ex = json.loads(line)
            q = ex.get('input', ex.get('question', ''))
            if q not in found_questions:
                missing_indices.append(i + 1) # +1 for 1-based display

print(f"\nIndices missing from the final 188-sample pool (1-based): {missing_indices}")

Pre-crash list (151-162 attempt) count: 0
Post-crash list (163-200 attempt) count: 38

Indices missing from the final 188-sample pool (1-based): [151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162]


In [ ]:
# ── Accuracy report for novelty (Samples 151-200 - Final) ───────────────
res_200_iters, res_200_accs = build_accuracy(novelty_results_151_200_final, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Final Samples 151-200)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_200_iters):
    acc = res_200_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_151_200_final)/100)):<10}{len(novelty_results_151_200_final):<10}{acc:.1f}%")


┃ NOVELTY SLICE ACCURACY (Final Samples 151-200)
Iter      Correct   Total     Accuracy    
---------------------------------------------
Base      31        38        81.6%
Iter 1    35        38        92.1%
Iter 2    35        38        92.1%
Iter 3    38        38        100.0%
Iter 4    38        38        100.0%


In [ ]:
# 1. Run the missing slice (indices 150 to 162 covers 151-162 in 1-based naming)
missing_slice_results = run_novelty_slice(GSM_FILE, 150, 162, MAX_ITER)

# 2. Consolidate ALL results into the final N=200 pool
# We use the previously verified results for 1-50, 51-100, 101-150, and 163-200
final_200_results = (
    novelty_results +
    novelty_results_51_100 +
    novelty_results_101_150 +
    missing_slice_results +
    novelty_results_163_200
)

# 3. Generate Final Report
final_200_iters, final_200_accs = build_accuracy(final_200_results, MAX_ITER)

print('=' * 55)
print(f'   FINAL CONSOLIDATED TABLE (N={len(final_200_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_200_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_200_accs[i] / 100) * len(final_200_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_200_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Gain: {final_200_accs[-1] - final_200_accs[0]:+.1f}%')
print('=' * 55)

📊 NOVELTY SLICE | Samples 150-162 | k=4

📌 [151/162] GT=4.0 | Steve and Tim decide to see who can get home from school the...
  ✅ iter 0  pred=4.0

📌 [152/162] GT=5.0 | Shawnda decides that her neighborhood kids could really use ...
  ✅ iter 0  pred=5.0

📌 [153/162] GT=4.0 | Carl buys ten packs of cookies. Each pack of cookies has six...
  ✅ iter 0  pred=4.0

📌 [154/162] GT=48.0 | Dave bought a large pack of french fries and ate fourteen be...
  ❌ iter 0  pred=38.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=48.0

📌 [155/162] GT=272.0 | On Mondays, Wednesdays, and Fridays, college student Kimo ha...
  ✅ iter 0  pred=272.0

📌 [156/162] GT=280.0 | Bill bakes 300 rolls, 120 chocolate croissants, and 60 bague...
  ✅ iter 0  pred=280.0

📌 [157/162] GT=1400.0 | The zookeeper feeds all the apes in the zoo. He orders all t...
  ❌ iter 0  pred=1600.0
    📝 [M

In [ ]:
# Run for samples 201 to 250
novelty_results_201_250 = run_novelty_slice(GSM_FILE, 200, 250, MAX_ITER)

# Consolidate the entire dataset up to N=250
final_250_results = (
    final_200_results +
    novelty_results_201_250
)

# Accuracy report for the new batch
res_250_iters, res_250_accs = build_accuracy(novelty_results_201_250, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 201-250)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_250_iters):
    acc = res_250_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_201_250)/100)):<10}{len(novelty_results_201_250):<10}{acc:.1f}%")

# Total Consolidation Report
final_250_iters, final_250_accs = build_accuracy(final_250_results, MAX_ITER)
print(f'\n✅ FINAL CONSOLIDATION (N={len(final_250_results)})')
print(f'Base Accuracy: {final_250_accs[0]:.1f}% | Final Accuracy: {final_250_accs[-1]:.1f}%')

📊 NOVELTY SLICE | Samples 200-250 | k=4

📌 [201/250] GT=55.0 | Baldur gets water from a well. He gets 5 pails of water ever...
  ✅ iter 0  pred=55.0

📌 [202/250] GT=114200.0 | John wins an award at work.  The award has a 1 time monetary...
  ✅ iter 0  pred=114200.0

📌 [203/250] GT=100.0 | Josie grows grapes on her 10-acre farm.  Each acre produces ...
  ✅ iter 0  pred=100.0

📌 [204/250] GT=31.0 | Carl’s favorite food is cheese. He ate a sandwich every day ...
  ✅ iter 0  pred=31.0

📌 [205/250] GT=98.0 | Janet had 22 green pens and 10 yellow pens. Then she bought ...
  ✅ iter 0  pred=98.0

📌 [206/250] GT=98.0 | Brinley is in Mr. Bert's math class. Mr. Bert gives six test...
  ❌ iter 0  pred=120.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ✅ iter 1  pred=98.0

📌 [207/250] GT=860.0 | Micheal loves riding a bike. He rode it at least 5 times a w...
  ✅ iter 0  pred=860

In [ ]:
# ── Run for samples 251 to 300 ──
novelty_results_251_300 = run_novelty_slice(GSM_FILE, 250, 300, MAX_ITER)

# Consolidate all 300 results
final_300_results = (
    final_250_results +
    novelty_results_251_300
)

# Accuracy report for the new batch
res_300_iters, res_300_accs = build_accuracy(novelty_results_251_300, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 251-300)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_300_iters):
    acc = res_300_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_251_300)/100)):<10}{len(novelty_results_251_300):<10}{acc:.1f}%")

# ── Final N=300 Consolidation ──
final_300_iters, final_300_accs = build_accuracy(final_300_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_300_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_300_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_300_accs[i] / 100) * len(final_300_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_300_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_300_accs[-1] - final_300_accs[0]:+.1f}%')
print('=' * 55)

📊 NOVELTY SLICE | Samples 250-300 | k=4

📌 [251/300] GT=17.0 | Each class in a school has 20 students. There are 3 classes....
  ❌ iter 0  pred=20.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=17.0

📌 [252/300] GT=70.0 | Travis had 61 apps on his tablet. He deleted 9 apps he didn'...
  ✅ iter 0  pred=70.0

📌 [253/300] GT=73.0 | Last night Rick killed ten wolves and 15 cougars while hunti...
  ✅ iter 0  pred=73.0

📌 [254/300] GT=18.0 | Bill starts on the 3rd floor. He rides the elevator up to th...
  ✅ iter 0  pred=18.0

📌 [255/300] GT=84.0 | Shelly's 3 kids spent all day at the water park.  Mitchel we...
  ✅ iter 0  pred=84.0

📌 [256/300] GT=192.0 | Ten stalls have 20 cows each. Mr. Sylas buys 40 cows and div...
  ❌ iter 0  pred=96.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2

In [ ]:
# ── Run for samples 301 to 350 ──
novelty_results_301_350 = run_novelty_slice(GSM_FILE, 300, 350, MAX_ITER)

# Consolidate all 350 results
final_350_results = (
    final_300_results +
    novelty_results_301_350
)

# Accuracy report for the new batch
res_350_iters, res_350_accs = build_accuracy(novelty_results_301_350, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 301-350)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_350_iters):
    acc = res_350_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_301_350)/100)):<10}{len(novelty_results_301_350):<10}{acc:.1f}%")

# ── Final N=350 Consolidation ──
final_350_iters, final_350_accs = build_accuracy(final_350_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_350_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_350_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_350_accs[i] / 100) * len(final_350_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_350_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_350_accs[-1] - final_350_accs[0]:+.1f}%')
print('=' * 55)


📊 NOVELTY SLICE | Samples 300-350 | k=4

📌 [301/350] GT=78.0 | On Tuesday, Peter wants to exercise for twice the amount of ...
  ✅ iter 0  pred=78.0

📌 [302/350] GT=8.0 | A simple folding newspaper or tabloid can be made by folding...
  ❌ iter 0  pred=32.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=8.0

📌 [303/350] GT=15.0 | Annika brought $50 to the town fair. She spent half of it on...
  ✅ iter 0  pred=15.0

📌 [304/350] GT=1300.0 | Elise has been selling her Dad's collection of 250 books for...
  ❌ iter 0  pred=4000.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ✅ iter 1  pred=1300.0

📌 [305/350] GT=3200.0 | Watson works a 10-hour shift each day, five days a week. He ...
  ✅ iter 0  pred=3200.0

📌 [306/350] GT=4.0 | John arm wr

In [ ]:
# ── Run for samples 351 to 400 ──
novelty_results_351_400 = run_novelty_slice(GSM_FILE, 350, 400, MAX_ITER)

# Consolidate all 400 results
final_400_results = (
    final_350_results +
    novelty_results_351_400
)

# Accuracy report for the new batch
res_400_iters, res_400_accs = build_accuracy(novelty_results_351_400, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 351-400)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_400_iters):
    acc = res_400_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_351_400)/100)):<10}{len(novelty_results_351_400):<10}{acc:.1f}%")

# ── Final N=400 Consolidation ──
final_400_iters, final_400_accs = build_accuracy(final_400_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_400_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_400_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_400_accs[i] / 100) * len(final_400_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_400_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_400_accs[-1] - final_400_accs[0]:+.1f}%')
print('=' * 55)

📊 NOVELTY SLICE | Samples 350-400 | k=4

📌 [351/400] GT=8.0 | Peyton scheduled after-work activities of a one hour yoga cl...
  ✅ iter 0  pred=8.0

📌 [352/400] GT=10.0 | April is donating plant pots to a local school for their new...
  ✅ iter 0  pred=10.0

📌 [353/400] GT=21.0 | After Andrea saved some money, she then spent the rest of he...
  ✅ iter 0  pred=21.0

📌 [354/400] GT=20.0 | John decides to do several activities while out on vacation....
  ❌ iter 0  pred=6.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=20.0

📌 [355/400] GT=45.0 | Annabelle is saving for a phone that costs $400. She already...
  ✅ iter 0  pred=45.0

📌 [356/400] GT=34.0 | There are three trees in Eddy's backyard. The shortest tree ...
  ✅ iter 0  pred=34.0

📌 [357/400] GT=21.0 | Dean's mother gave him $28 to go to the toy store. Dean boug...
  ✅ iter 0  pred=21.0

📌 [358/400] 

In [ ]:
# ── Run for samples 401 to 450 ──
novelty_results_401_450 = run_novelty_slice(GSM_FILE, 400, 450, MAX_ITER)

# Consolidate all 450 results
final_450_results = (
    final_400_results +
    novelty_results_401_450
)

# Accuracy report for the new batch
res_450_iters, res_450_accs = build_accuracy(novelty_results_401_450, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 401-450)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_450_iters):
    acc = res_450_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_401_450)/100)):<10}{len(novelty_results_401_450):<10}{acc:.1f}%")

# ── Final N=450 Consolidation ──
final_450_iters, final_450_accs = build_accuracy(final_450_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_450_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_450_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_450_accs[i] / 100) * len(final_450_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_450_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_450_accs[-1] - final_450_accs[0]:+.1f}%')
print('=' * 55)

📊 NOVELTY SLICE | Samples 400-450 | k=4

📌 [401/450] GT=48.0 | A glass of milk is 8 ounces of milk.  John drinks 2 glasses ...
  ✅ iter 0  pred=48.0

📌 [402/450] GT=14400.0 | A builder works for 4 weeks every month and for 6 days every...
  ✅ iter 0  pred=14400.0

📌 [403/450] GT=4.0 | Mark is making a quadruple batch of brownies. The normal rec...
  ✅ iter 0  pred=4.0

📌 [404/450] GT=81.0 | Mel uses a 900-watt air conditioner for 8 hours a day. This ...
  ❌ iter 0  pred=1080.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ❌ iter 1  pred=135.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ❌ iter 2  pred=135.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
  

In [ ]:
# ── Run for samples 451 to 500 ──
novelty_results_451_500 = run_novelty_slice(GSM_FILE, 450, 500, MAX_ITER)

# Consolidate all 500 results
final_500_results = (
    final_450_results +
    novelty_results_451_500
)

# Accuracy report for the new batch
res_500_iters, res_500_accs = build_accuracy(novelty_results_451_500, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 451-500)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_500_iters):
    acc = res_500_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_451_500)/100)):<10}{len(novelty_results_451_500):<10}{acc:.1f}%")

# ── Final N=500 Consolidation ──
final_500_iters, final_500_accs = build_accuracy(final_500_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_500_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_500_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_500_accs[i] / 100) * len(final_500_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_500_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_500_accs[-1] - final_500_accs[0]:+.1f}%')
print('=' * 55)

📊 NOVELTY SLICE | Samples 450-500 | k=4

📌 [451/500] GT=4.0 | You can buy a movie super ticket for $20 that includes right...
  ❌ iter 0  pred=14.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=4.0

📌 [452/500] GT=11050.0 | On a certain day, the total cost of filling up 20 helium bal...
  ✅ iter 0  pred=11050.0

📌 [453/500] GT=50.0 | A car is on a road trip and drives 60 mph for 2 hours, and t...
  ✅ iter 0  pred=50.0

📌 [454/500] GT=6400.0 | Jenna starts out with 8 sapphires. She trades 3 sapphires fo...
  ❌ iter 0  pred=14800.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ✅ iter 1  pred=6400.0

📌 [455/500] GT=150.0 | Marin and his neighbor Nancy each eat 4 apples a day. How ma...
  ❌ iter 0  pred=240.0
    📝 [Model 2] Generating F

In [ ]:
# ── Run for samples 501 to 550 ──
novelty_results_501_550 = run_novelty_slice(GSM_FILE, 500, 550, MAX_ITER)

# Consolidate all 550 results
final_550_results = (
    final_500_results +
    novelty_results_501_550
)

# Accuracy report for the new batch
res_550_iters, res_550_accs = build_accuracy(novelty_results_501_550, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 501-550)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_550_iters):
    acc = res_550_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_501_550)/100)):<10}{len(novelty_results_501_550):<10}{acc:.1f}%")

# ── Final N=550 Consolidation ──
final_550_iters, final_550_accs = build_accuracy(final_550_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_550_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_550_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_550_accs[i] / 100) * len(final_550_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_550_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_550_accs[-1] - final_550_accs[0]:+.1f}%')
print('=' * 55)

# Save checkpoint
with open('/content/novelty_results_501_550.pkl', 'wb') as f:
    pickle.dump(novelty_results_501_550, f)

📊 NOVELTY SLICE | Samples 500-550 | k=4

📌 [501/550] GT=16.0 | Together Lily, David, and Bodhi collected 43 insects. Lily f...
  ❌ iter 0  pred=None
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ❌ iter 1  pred=19.285714285714285
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 2  pred=16.0

📌 [502/550] GT=273.0 | Mariah’s grandma was teaching her to knit. Mariah used 1/4 o...
  ✅ iter 0  pred=273.0

📌 [503/550] GT=26.0 | Cherrie wants to buy Christmas gifts for her 5 friends. 2 of...
  ✅ iter 0  pred=26.0

📌 [504/550] GT=18.0 | The rug is 5 feet wider than the chair. The couch is 2 feet ...
  ✅ iter 0  pred=18.0

📌 [505/550] GT=2.0 | Suzie loves to chew fruit-flavored gum. She bought four pack...
  ✅ iter 0  pred=2.0

📌 [506/550] GT=1600.0 | Fr

In [ ]:
# ── Run for samples 551 to 600 ──
novelty_results_551_600 = run_novelty_slice(GSM_FILE, 550, 600, MAX_ITER)

# Consolidate all 600 results
final_600_results = (
    final_550_results +
    novelty_results_551_600
)

# Accuracy report for the new batch
res_600_iters, res_600_accs = build_accuracy(novelty_results_551_600, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 551-600)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_600_iters):
    acc = res_600_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_551_600)/100)):<10}{len(novelty_results_551_600):<10}{acc:.1f}%")

# ── Final N=600 Consolidation ──
final_600_iters, final_600_accs = build_accuracy(final_600_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_600_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_600_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_600_accs[i] / 100) * len(final_600_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_600_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_600_accs[-1] - final_600_accs[0]:+.1f}%')
print('=' * 55)

# Save checkpoint
with open('/content/novelty_results_551_600.pkl', 'wb') as f:
    pickle.dump(novelty_results_551_600, f)

📊 NOVELTY SLICE | Samples 550-600 | k=4

📌 [551/600] GT=27.0 | A shop sells school supplies. One notebook is sold at $1.50 ...
  ✅ iter 0  pred=27.0

📌 [552/600] GT=17.0 | Carly wants to treat her friends. She orders five hamburgers...
  ✅ iter 0  pred=17.0

📌 [553/600] GT=450.0 | A marketing company pays its employees on a commission-based...
  ❌ iter 0  pred=900.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=450.0

📌 [554/600] GT=92.0 | Bill is trying to figure out how many water bottles he needs...
  ✅ iter 0  pred=92.0

📌 [555/600] GT=54.0 | When the water is cold Ray swims a mile in 16 minutes. When ...
  ✅ iter 0  pred=54.0

📌 [556/600] GT=2.0 | John plans to save money from working.  He gets paid $2 per ...
  ✅ iter 0  pred=2.0

📌 [557/600] GT=160.0 | How much does it cost you for lunch today at Subway if you p...
  ✅ iter 0  pred=160.0

📌 [558

In [ ]:
# ── Run for samples 601 to 650 ──
novelty_results_601_650 = run_novelty_slice(GSM_FILE, 600, 650, MAX_ITER)

# Consolidate all 650 results
final_650_results = (
    final_600_results +
    novelty_results_601_650
)

# Accuracy report for the new batch
res_650_iters, res_650_accs = build_accuracy(novelty_results_601_650, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 601-650)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_650_iters):
    acc = res_650_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_601_650)/100)):<10}{len(novelty_results_601_650):<10}{acc:.1f}%")

# ── Final N=650 Consolidation ──
final_650_iters, final_650_accs = build_accuracy(final_650_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_650_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_650_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_650_accs[i] / 100) * len(final_650_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_650_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_650_accs[-1] - final_650_accs[0]:+.1f}%')
print('=' * 55)

# Save checkpoint
with open('/content/novelty_results_601_650.pkl', 'wb') as f:
    pickle.dump(novelty_results_601_650, f)

📊 NOVELTY SLICE | Samples 600-650 | k=4

📌 [601/650] GT=10.0 | Jamaal is at the gym. He has been using an 8-pound weight. H...
  ✅ iter 0  pred=10.0

📌 [602/650] GT=104.0 | Steve loves playing video games.  His parents get him a cons...
  ❌ iter 0  pred=94.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=104.0

📌 [603/650] GT=5.0 | A plane travels 1200 miles in 3 hours. At the same rate, how...
  ❌ iter 0  pred=2.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=5.0

📌 [604/650] GT=1800.0 | Ruiz can make 120 pounds of chocolates in two hours. Marissa...
  ❌ iter 0  pred=1260.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    🏆 Skipped ranking — using Feedback A
  ✅ 

In [ ]:
# ── Run for samples 651 to 700 ──
novelty_results_651_700 = run_novelty_slice(GSM_FILE, 650, 700, MAX_ITER)

# Consolidate all 700 results
final_700_results = (
    final_650_results +
    novelty_results_651_700
)

# Accuracy report for the new batch
res_700_iters, res_700_accs = build_accuracy(novelty_results_651_700, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 651-700)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_700_iters):
    acc = res_700_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_651_700)/100)):<10}{len(novelty_results_651_700):<10}{acc:.1f}%")

# ── Final N=700 Consolidation ──
final_700_iters, final_700_accs = build_accuracy(final_700_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_700_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_700_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_700_accs[i] / 100) * len(final_700_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_700_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_700_accs[-1] - final_700_accs[0]:+.1f}%')
print('=' * 55)

# Save checkpoint
with open('/content/novelty_results_651_700.pkl', 'wb') as f:
    pickle.dump(novelty_results_651_700, f)


📊 NOVELTY SLICE | Samples 650-700 | k=4

📌 [651/700] GT=142.0 | On Tuesday, Clara bought 20 pomegranates at $20 each. At the...
  ✅ iter 0  pred=142.0

📌 [652/700] GT=2100.0 | Ariadne has a shop selling hats of two different colors, red...
  ❌ iter 0  pred=1400.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ✅ iter 1  pred=2100.0

📌 [653/700] GT=75.0 | James hires a horse-drawn carriage from 5 PM to 9 PM.  He ge...
  ❌ iter 0  pred=105.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ✅ iter 1  pred=75.0

📌 [654/700] GT=80.0 | Sally has realized she did not receive a full wage this week...
  ✅ iter 0  pred=80.0

📌 [655/700] GT=2.0 | Lori needed 1 whole egg to make 2 deviled egg halves.  She a...
  ❌ iter 0  pred=4.0
    📝 [Model 2] Generating Feedbac

In [ ]:
# ── Run for samples 701 to 750 ──
novelty_results_701_750 = run_novelty_slice(GSM_FILE, 700, 750, MAX_ITER)

# Save checkpoint
with open('/content/novelty_results_701_750.pkl', 'wb') as f:
    pickle.dump(novelty_results_701_750, f)

📊 NOVELTY SLICE | Samples 700-750 | k=4

📌 [701/750] GT=135.0 | Hannah's city is having a big display of fireworks for the 4...
  ✅ iter 0  pred=135.0

📌 [702/750] GT=200.0 | Aiden and 12 of his friends are going to see a film at the c...
  ✅ iter 0  pred=200.0

📌 [703/750] GT=2800.0 | Gissela, Gordy, and Gary are truck drivers.  Gissela has a t...
  ✅ iter 0  pred=2800.0

📌 [704/750] GT=50.0 | Larry cooked dumplings for a group of friends.  There are 8 ...
  ✅ iter 0  pred=50.0

📌 [705/750] GT=50.0 | Gerald and Julia divided $100 in the ratio 3:2. If Gerald sp...
  ✅ iter 0  pred=50.0

📌 [706/750] GT=120.0 | Martha's cat is 5 times faster than her turtle. If the cat c...
  ✅ iter 0  pred=120.0

📌 [707/750] GT=9.0 | The local firefighters are doing a “fill the boot” fundraise...
  ✅ iter 0  pred=9.0

📌 [708/750] GT=8.0 | Colorado City uses 40% of the water from the Colorado River....
  ✅ iter 0  pred=8.0

📌 [709/750] GT=168.0 | Given a 7-day week, how much does Alex charge for 2 weeks 

In [ ]:
# ── Run for samples 751 to 800 ──
novelty_results_751_800 = run_novelty_slice(GSM_FILE, 750, 800, MAX_ITER)

# Save checkpoint
with open('/content/novelty_results_751_800.pkl', 'wb') as f:
    pickle.dump(novelty_results_751_800, f)

📊 NOVELTY SLICE | Samples 750-800 | k=4

📌 [751/800] GT=4000.0 | Candy has a chair rental business. During the weekdays, 60 c...
  ❌ iter 0  pred=2000.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=4000.0

📌 [752/800] GT=43.0 | Gunther, the gorilla, had 48 bananas hidden under a fern bra...
  ✅ iter 0  pred=43.0

📌 [753/800] GT=240.0 | Jenna has 4 roommates. Each month the electricity bill is $1...
  ❌ iter 0  pred=300.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ✅ iter 1  pred=240.0

📌 [754/800] GT=128.0 | Jeff owns a catering company.  During a recent event, he sen...
  ❌ iter 0  pred=None
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …


KeyboardInterrupt: 

In [ ]:
# Consolidate all 800 results
final_800_results = (
    final_700_results +
    novelty_results_701_750 +
    novelty_results_751_800
)

# Accuracy report for the new 100-sample batch (701-800)
new_batch_results = novelty_results_701_750 + novelty_results_751_800
res_800_slice_iters, res_800_slice_accs = build_accuracy(new_batch_results, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 701-800)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_800_slice_iters):
    acc = res_800_slice_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(new_batch_results)/100)):<10}{len(new_batch_results):<10}{acc:.1f}%")

# ── Final N=800 Consolidation ──
final_800_iters, final_800_accs = build_accuracy(final_800_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_800_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_800_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_800_accs[i] / 100) * len(final_800_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_800_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_800_accs[-1] - final_800_accs[0]:+.1f}%')
print('=' * 55)


┃ NOVELTY SLICE ACCURACY (Samples 701-800)
Iter      Correct   Total     Accuracy    
---------------------------------------------
Base      81        100       81.0%
Iter 1    94        100       94.0%
Iter 2    95        100       95.0%
Iter 3    95        100       95.0%
Iter 4    96        100       96.0%

   CONSOLIDATED ACCURACY TABLE (N=800)
Iteration          | Correct    | Accuracy  
-------------------------------------------------------
Base (Iter 0)      | 619        | 77.4%
Iteration 1        | 758        | 94.8%
Iteration 2        | 771        | 96.4%
Iteration 3        | 776        | 97.0%
Iteration 4        | 778        | 97.2%
-------------------------------------------------------
Total Improvement: +19.9%


In [ ]:
# ── Run for samples 801 to 850 ──
novelty_results_801_850 = run_novelty_slice(GSM_FILE, 800, 850, MAX_ITER)

# Save checkpoint
with open('/content/novelty_results_801_850.pkl', 'wb') as f:
    pickle.dump(novelty_results_801_850, f)

📊 NOVELTY SLICE | Samples 800-850 | k=4

📌 [801/850] GT=428.0 | Pierson scored 278 points in one game of bowling. Nikita sco...
  ❌ iter 0  pred=1946.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=428.0

📌 [802/850] GT=1240.0 | At Ashley's school, they start a reforestation campaign wher...
  ⏳ Rate limit [llama-3.1-8b-instant] — waiting 35s (attempt 1)…
  ✅ iter 0  pred=1240.0

📌 [803/850] GT=6.0 | Bubbles collects stuffed animals. She has three stuffed pupp...
  ⏳ Rate limit [llama-3.1-8b-instant] — waiting 35s (attempt 1)…
  ❌ iter 0  pred=4.2
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=6.0

📌 [804/850] GT=9.0 | Kris is trying to earn a video game achievement for playing ...
  ✅ iter 0  pred=9.0

📌 [805/850] GT=2

In [ ]:
# Consolidate all 850 results
final_850_results = (
    final_800_results +
    novelty_results_801_850
)

# Accuracy report for the new batch (801-850)
res_850_slice_iters, res_850_slice_accs = build_accuracy(novelty_results_801_850, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 801-850)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_850_slice_iters):
    acc = res_850_slice_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_801_850)/100)):<10}{len(novelty_results_801_850):<10}{acc:.1f}%")

# ── Final N=850 Consolidation ──
final_850_iters, final_850_accs = build_accuracy(final_850_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_850_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_850_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_850_accs[i] / 100) * len(final_850_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_850_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_850_accs[-1] - final_850_accs[0]:+.1f}%')
print('=' * 55)


┃ NOVELTY SLICE ACCURACY (Samples 801-850)
Iter      Correct   Total     Accuracy    
---------------------------------------------
Base      38        50        76.0%
Iter 1    45        50        90.0%
Iter 2    46        50        92.0%
Iter 3    46        50        92.0%
Iter 4    46        50        92.0%

   CONSOLIDATED ACCURACY TABLE (N=850)
Iteration          | Correct    | Accuracy  
-------------------------------------------------------
Base (Iter 0)      | 657        | 77.3%
Iteration 1        | 803        | 94.5%
Iteration 2        | 817        | 96.1%
Iteration 3        | 822        | 96.7%
Iteration 4        | 824        | 96.9%
-------------------------------------------------------
Total Improvement: +19.6%


In [ ]:
# ── Run for samples 851 to 900 ──
novelty_results_851_900 = run_novelty_slice(GSM_FILE, 850, 900, MAX_ITER)

# Save checkpoint
with open('/content/novelty_results_851_900.pkl', 'wb') as f:
    pickle.dump(novelty_results_851_900, f)

📊 NOVELTY SLICE | Samples 850-900 | k=4

📌 [851/900] GT=70.0 | Mr. Smith has two farms, Farm X and Farm Y. He has 55 goats ...
  ✅ iter 0  pred=70.0

📌 [852/900] GT=110.0 | James buys 2 pairs of shoes a month.  He spends $2640 on sho...
  ❌ iter 0  pred=2.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ✅ iter 1  pred=110.0

📌 [853/900] GT=123.0 | A basket of green food costs $25 and a basket of red food co...
  ✅ iter 0  pred=123.0

📌 [854/900] GT=15.0 | There are 90 rooms at the KozyInn Motel. It takes housekeepi...
  ✅ iter 0  pred=15.0

📌 [855/900] GT=144.0 | A local town is expanding and wants to build several new hom...
  ❌ iter 0  pred=120.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ✅ iter 1  pred=144.0

📌 [856/900] GT=13.0 | I am three y

In [ ]:
# Consolidate all 900 results
final_900_results = (
    final_850_results +
    novelty_results_851_900
)

# Accuracy report for the new batch (851-900)
res_900_slice_iters, res_900_slice_accs = build_accuracy(novelty_results_851_900, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 851-900)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_900_slice_iters):
    acc = res_900_slice_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_851_900)/100)):<10}{len(novelty_results_851_900):<10}{acc:.1f}%")

# ── Final N=900 Consolidation ──
final_900_iters, final_900_accs = build_accuracy(final_900_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_900_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_900_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_900_accs[i] / 100) * len(final_900_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_900_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_900_accs[-1] - final_900_accs[0]:+.1f}%')
print('=' * 55)


┃ NOVELTY SLICE ACCURACY (Samples 851-900)
Iter      Correct   Total     Accuracy    
---------------------------------------------
Base      39        50        78.0%
Iter 1    48        50        96.0%
Iter 2    49        50        98.0%
Iter 3    49        50        98.0%
Iter 4    49        50        98.0%

   CONSOLIDATED ACCURACY TABLE (N=900)
Iteration          | Correct    | Accuracy  
-------------------------------------------------------
Base (Iter 0)      | 696        | 77.3%
Iteration 1        | 851        | 94.6%
Iteration 2        | 866        | 96.2%
Iteration 3        | 871        | 96.8%
Iteration 4        | 873        | 97.0%
-------------------------------------------------------
Total Improvement: +19.7%


In [ ]:
# ── Run for samples 901 to 950 ──
novelty_results_901_950 = run_novelty_slice(GSM_FILE, 900, 950, MAX_ITER)

# Save checkpoint
with open('/content/novelty_results_901_950.pkl', 'wb') as f:
    pickle.dump(novelty_results_901_950, f)

📊 NOVELTY SLICE | Samples 900-950 | k=4

📌 [901/950] GT=15.0 | Denise and Daniel are reading the same book. Yesterday, Deni...
  ❌ iter 0  pred=20.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=15.0

📌 [902/950] GT=1.0 | Calvin is making soup for his family for dinner. He has a po...
  ✅ iter 0  pred=1.0

📌 [903/950] GT=8.0 | The educational shop is selling notebooks for $1.50 each and...
  ✅ iter 0  pred=8.0

📌 [904/950] GT=16.0 | Jo has been making face masks. She can make 4 small masks wi...
  ✅ iter 0  pred=16.0

📌 [905/950] GT=8.0 | There are 9 Fast and the Furious movies, Deepa has seen each...
  ✅ iter 0  pred=8.0

📌 [906/950] GT=5.0 | Harold sleeps for 10 hours a night.  He works 2 hours less t...
  ❌ iter 0  pred=17.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Rankin

In [ ]:
# Consolidate all 950 results
final_950_results = (
    final_900_results +
    novelty_results_901_950
)

# Accuracy report for the new batch (901-950)
res_950_slice_iters, res_950_slice_accs = build_accuracy(novelty_results_901_950, MAX_ITER)

print('\n' + '=' * 65)
print('┃ NOVELTY SLICE ACCURACY (Samples 901-950)')
print('=' * 65)
print(f"{'Iter':<10}{'Correct':<10}{'Total':<10}{'Accuracy':<12}")
print('-' * 45)

for i, it in enumerate(res_950_slice_iters):
    acc = res_950_slice_accs[i]
    label = 'Base' if it == 0 else f'Iter {it}'
    print(f"{label:<10}{int(round(acc*len(novelty_results_901_950)/100)):<10}{len(novelty_results_901_950):<10}{acc:.1f}%")

# ── Final N=950 Consolidation ──
final_950_iters, final_950_accs = build_accuracy(final_950_results, MAX_ITER)

print('\n' + '=' * 55)
print(f'   CONSOLIDATED ACCURACY TABLE (N={len(final_950_results)})')
print('=' * 55)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 55)

for i, it in enumerate(final_950_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_950_accs[i] / 100) * len(final_950_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_950_accs[i]:.1f}%')

print('-' * 55)
print(f'Total Improvement: {final_950_accs[-1] - final_950_accs[0]:+.1f}%')
print('=' * 55)


┃ NOVELTY SLICE ACCURACY (Samples 901-950)
Iter      Correct   Total     Accuracy    
---------------------------------------------
Base      38        50        76.0%
Iter 1    48        50        96.0%
Iter 2    48        50        96.0%
Iter 3    48        50        96.0%
Iter 4    48        50        96.0%

   CONSOLIDATED ACCURACY TABLE (N=950)
Iteration          | Correct    | Accuracy  
-------------------------------------------------------
Base (Iter 0)      | 734        | 77.3%
Iteration 1        | 899        | 94.6%
Iteration 2        | 914        | 96.2%
Iteration 3        | 919        | 96.7%
Iteration 4        | 921        | 96.9%
-------------------------------------------------------
Total Improvement: +19.7%


In [ ]:
# ── Run for the next batch: samples 1001 to 1100 ──
novelty_results_1001_1100 = run_novelty_slice(GSM_FILE, 1000, 1100, MAX_ITER)

# Save checkpoint
with open('/content/novelty_results_1001_1100.pkl', 'wb') as f:
    pickle.dump(novelty_results_1001_1100, f)

# ── Final Consolidation (N=1100) ──
final_1100_results = final_1000_results + novelty_results_1001_1100
final_1100_iters, final_1100_accs = build_accuracy(final_1100_results, MAX_ITER)

print('\n' + '=' * 65)
print('┃ CONSOLIDATED NOVELTY REPORT (N=1100)')
print('=' * 65)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 45)

for i, it in enumerate(final_1100_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_1100_accs[i] / 100) * len(final_1100_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_1100_accs[i]:.1f}%')

print('-' * 45)
print(f'Total Absolute Improvement: {final_1100_accs[-1] - final_1100_accs[0]:+.1f}%')
print('=' * 65)

📊 NOVELTY SLICE | Samples 1000-1100 | k=4

📌 [1001/1100] GT=1.0 | Doctor Jones is scheduling his time for Monday. He is spendi...
  ❌ iter 0  pred=-86.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=1.0

📌 [1002/1100] GT=2.0 | Jordan wanted to surprise her mom with a homemade birthday c...
  ❌ iter 0  pred=840.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ❌ iter 1  pred=None
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ❌ iter 2  pred=None
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ❌ iter 3 

In [ ]:
# ── Run for the next batch: samples 1101 to 1200 ──
novelty_results_1101_1200 = run_novelty_slice(GSM_FILE, 1100, 1200, MAX_ITER)

# Save checkpoint
with open('/content/novelty_results_1101_1200.pkl', 'wb') as f:
    pickle.dump(novelty_results_1101_1200, f)

# ── Final Consolidation (N=1200) ──
final_1200_results = final_1100_results + novelty_results_1101_1200
final_1200_iters, final_1200_accs = build_accuracy(final_1200_results, MAX_ITER)

print('\n' + '=' * 65)
print('┃ CONSOLIDATED NOVELTY REPORT (N=1200)')
print('=' * 65)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 45)

for i, it in enumerate(final_1200_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_1200_accs[i] / 100) * len(final_1200_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_1200_accs[i]:.1f}%')

print('-' * 45)
print(f'Total Absolute Improvement: {final_1200_accs[-1] - final_1200_accs[0]:+.1f}%')
print('=' * 65)

📊 NOVELTY SLICE | Samples 1100-1200 | k=4

📌 [1101/1200] GT=5.0 | Bobby takes a 30 min lunch and 2 15 minutes break per day at...
  ✅ iter 0  pred=5.0

📌 [1102/1200] GT=16.0 | Amber, Micah, and Ahito ran 52 miles in total. Amber ran 8 m...
  ✅ iter 0  pred=16.0

📌 [1103/1200] GT=113.0 | Sheila charged $85.00 worth of merchandise on her credit car...
  ✅ iter 0  pred=113.0

📌 [1104/1200] GT=90.0 | A jellyfish tank has numerous jellyfish in it. A fifth of th...
  ✅ iter 0  pred=90.0

📌 [1105/1200] GT=24.0 | Caroline is three times older than Ben. Ben is two times old...
  ✅ iter 0  pred=24.0

📌 [1106/1200] GT=40.0 | Lauren is saving 20% of every paycheck. How many more years ...
  ❌ iter 0  pred=249.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=40.0

📌 [1107/1200] GT=5.0 | Marty has 100 centimeters of ribbon that he must cut into 4 ...
  ✅ iter 0  pred

In [ ]:
# ── Run for samples 1201 to 1310 ──
novelty_results_1201_1310 = run_novelty_slice(GSM_FILE, 1200, 1310, MAX_ITER)

# Save checkpoint
with open('/content/novelty_results_1201_1310.pkl', 'wb') as f:
    pickle.dump(novelty_results_1201_1310, f)

# ── Final Consolidation (N=1310) ──
final_1310_results = final_1200_results + novelty_results_1201_1310
final_1310_iters, final_1310_accs = build_accuracy(final_1310_results, MAX_ITER)

print('\n' + '=' * 65)
print('┃ CONSOLIDATED NOVELTY REPORT (N=1310)')
print('=' * 65)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 45)

for i, it in enumerate(final_1310_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_1310_accs[i] / 100) * len(final_1310_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_1310_accs[i]:.1f}%')

print('-' * 45)
print(f'Total Absolute Improvement: {final_1310_accs[-1] - final_1310_accs[0]:+.1f}%')
print('=' * 65)

📊 NOVELTY SLICE | Samples 1200-1310 | k=4

📌 [1201/1310] GT=8.0 | Eight more than four times the number of coffee mugs in the ...
  ✅ iter 0  pred=8.0

📌 [1202/1310] GT=42.0 | There are 66 fish in the fish tank. One-third of the fish ha...
  ✅ iter 0  pred=42.0

📌 [1203/1310] GT=19.0 | Amy had two eyeshadow palettes with four colors each and thr...
  ❌ iter 0  pred=13.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …


KeyboardInterrupt: 

In [ ]:
# ── Run for the final batch: samples 1311 to 1319 ──
novelty_results_1311_1319 = run_novelty_slice(GSM_FILE, 1310, 1319, MAX_ITER)

# Save checkpoint
with open('/content/novelty_results_1311_1319.pkl', 'wb') as f:
    pickle.dump(novelty_results_1311_1319, f)

# ── Final Consolidation (N=1319) ──
final_1319_results = final_1310_results + novelty_results_1311_1319
final_1319_iters, final_1319_accs = build_accuracy(final_1319_results, MAX_ITER)

print('\n' + '=' * 65)
print('┃ FINAL CONSOLIDATED NOVELTY REPORT (N=1319)')
print('=' * 65)
print(f"{'Iteration':<18} | {'Correct':<10} | {'Accuracy':<10}")
print('-' * 45)

for i, it in enumerate(final_1319_iters):
    label = 'Base (Iter 0)' if it == 0 else f'Iteration {it}'
    correct_count = int(round((final_1319_accs[i] / 100) * len(final_1319_results)))
    print(f'{label:<18} | {correct_count:<10} | {final_1319_accs[i]:.1f}%')

print('-' * 45)
print(f'Total Absolute Improvement: {final_1319_accs[-1] - final_1319_accs[0]:+.1f}%')
print('=' * 65)

📊 NOVELTY SLICE | Samples 1310-1319 | k=4

📌 [1311/1319] GT=64.0 | Aaron and Vanessa were relay race partners on a running team...
  ✅ iter 0  pred=64.0

📌 [1312/1319] GT=594.0 | The caretaker of the docks needs to buy some new line. He wa...
  ❌ iter 0  pred=603.0
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback B
  ✅ iter 1  pred=594.0

📌 [1313/1319] GT=180.0 | Tom's restaurant gets 6 reservations a night.  They normally...
  ✅ iter 0  pred=180.0

📌 [1314/1319] GT=2.0 | A fruit vendor bought 50 watermelons for $80. He sold all of...
  ❌ iter 0  pred=2.1333333333333333
    📝 [Model 2] Generating Feedback A (temp=0.0) …
    📝 [Model 2] Generating Feedback B (temp=0.4) …
    ⚖️  [Model 2] Ranking feedbacks …
    🏆 Winner: Feedback A
  ✅ iter 1  pred=2.0

📌 [1315/1319] GT=8.0 | John had a son James when he was 19.  James is now twice as ...
  ❌ iter 0  pred=35.0
    📝 [Model 

In [ ]:
import pickle
from google.colab import files

# The final consolidated list exists in 'final_1319_results'
FILENAME = 'novelty_results_final_1319.pkl'

# Save the list to a pickle file
with open(FILENAME, 'wb') as f:
    pickle.dump(final_1319_results, f)

print(f'✅ Results saved to {FILENAME}')

# Download the file to local storage
files.download(FILENAME)

✅ Results saved to novelty_results_final_1319.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
all_consolidated_winners = [w for ex in final_1319_results for w in ex.get('fb_winners', [])]

a_total = all_consolidated_winners.count('A')
b_total = all_consolidated_winners.count('B')
total_decisions = len(all_consolidated_winners)

print(f'📊 TOTAL FEEDBACK USAGE (N=1319)')
print('='*35)
print(f'Feedback A (Deterministic) wins: {a_total} ({a_total/total_decisions*100:.1f}%)')
print(f'Feedback B (Exploratory) wins:   {b_total} ({b_total/total_decisions*100:.1f}%)')
print(f'Total Ranking Decisions:        {total_decisions}')

📊 TOTAL FEEDBACK USAGE (N=1319)
Feedback A (Deterministic) wins: 305 (61.2%)
Feedback B (Exploratory) wins:   193 (38.8%)
Total Ranking Decisions:        498


## Why three models?

| Model | Role | Rationale |
|---|---|---|
| `llama-3.1-8b-instant` | **Initial generation** | Fast turnaround, reliable Python code synthesis |
| `gemma2-9b-it` | **Dual feedback + ranking** | Google's Gemma 2 excels at analytical evaluation and natural-language reasoning |
| `mixtral-8x7b-32768` | **Refinement** | Mixtral's MoE architecture gives strong reasoning for complex bug-fixing |

## Why dual feedback?

A single feedback prompt can miss errors or be vague. By generating **two independent feedbacks**
(one deterministic at temperature 0, one slightly exploratory at temperature 0.4), and asking the
feedback model to **rank** them, we consistently select the more actionable critique — leading to
a higher-quality refinement step.

## Rate-limit advantage

Free-tier Groq accounts have per-model quotas. Splitting calls across three distinct models
effectively **triples** the available request budget compared to a single-model approach.
